## - Construindo o pipeline da câmera

Vamos desenvolver um pipeline que parte de um RAW real e termina em JPEG, implementando passos para executar o demosaicing, balanço de branco, redução de ruído, tons, nitidez e compressão.

O notebook deve ser preenchido com todos as partes requisitadas e marcadas como 'TODO' ao longo do código. As operações devem ser executadas em ordem, não altere os nomes das variavéis ou exibição que foram definidas a priori.

O RAW contém amostras do sensor e metadados, não um RGB final. O pacote rawpy/LibRaw será usado para decodificação e parte do processamento de baixo nível. 

In [ ]:
!pip install rawpy
!pip install opencv-python

from pathlib import Path
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import rawpy

PASTA_SAIDA = Path("resultados_raw")
PASTA_SAIDA.mkdir(exist_ok=True)
plt.rcParams["figure.figsize"] = (12, 5)
print("LibRaw:", rawpy.libraw_version, "| OpenCV:", cv2.__version__)

def exibir(pares, cmap=None):
    pares = [(titulo, imagem) for titulo, imagem in pares if imagem is not None]
    fig, eixos = plt.subplots(1, len(pares), figsize=(5 * len(pares), 5))
    for ax, (titulo, imagem) in zip(np.atleast_1d(eixos), pares):
        ax.imshow(imagem, cmap=cmap if imagem.ndim == 2 else None)
        ax.set_title(titulo)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

def linear_para_srgb8(imagem_linear):
    '''Codificação sRGB para visualização de RGB linear normalizado.'''
    x = np.clip(np.asarray(imagem_linear, dtype=np.float32), 0.0, 1.0)
    srgb = np.where(x <= 0.0031308, 12.92 * x,
                    1.055 * np.power(x, 1.0 / 2.4) - 0.055)
    return np.rint(np.clip(srgb, 0.0, 1.0) * 255.0).astype(np.uint8)


### 1 - Carregamento e análise do RAW

In [ ]:
#TODO: (1) Passe o arquivo RAW para leitura
CAMINHO_RAW = Path("arquivo_de_imagem_raw.dng")

if not CAMINHO_RAW.is_file():
    raise FileNotFoundError(f"Defina CAMINHO_RAW: {CAMINHO_RAW}")

with rawpy.imread(str(CAMINHO_RAW)) as raw:
    mosaico_raw = raw.raw_image_visible.copy()
    indices_cores = raw.raw_colors_visible.copy()
    padrao = None if raw.raw_pattern is None else raw.raw_pattern.copy()
    descricao_cores = raw.color_desc.decode("ascii", errors="replace")
    preto_por_canal = list(raw.black_level_per_channel)
    branco_global = int(raw.white_level)
    branco_por_canal = raw.camera_white_level_per_channel
    wb_camera = list(raw.camera_whitebalance)
    previa_rawpy = raw.postprocess(
        use_camera_wb=True, no_auto_bright=True,
        output_color=rawpy.ColorSpace.sRGB, output_bps=8
    )

if mosaico_raw.ndim != 2 or padrao is None or padrao.shape != (2, 2):
    raise ValueError("Este roteiro requer RAW Bayer plano com padrão 2×2.")

print("Arquivo:", CAMINHO_RAW.name)
print("Mosaico:", mosaico_raw.shape, mosaico_raw.dtype)
print("Arquivo MB:", round(CAMINHO_RAW.stat().st_size / 1_000_000, 2))
print("Padrão de índices:\n", padrao, "| Cores:", descricao_cores)
print("Preto:", preto_por_canal, "| Branco:", branco_por_canal or branco_global)
print("WB da câmera:", wb_camera)
exibir([("Prévia rawpy", previa_rawpy)])

In [ ]:
#TODO: (2) Implemente a função que traduza os indices armazenados em padrao, usando descricao_cores
def get_padrao_legivel(padrao, descricao_cores):
    pass
    padrao_legivel = None
    return padrao_legivel

print("Padrão CFA:\n", get_padrao_legivel(padrao, descricao_cores))

# TODO: (3) Calcule mínimo, máximo e percentual de amostras próximas ao branco, substitua os valores None pelos correspondentes
amostras = mosaico_raw.ravel()[::max(1, mosaico_raw.size // 200_000)]

estatisticas_raw = {
    "minimo": None,
    "maximo": None,
    "media": None,
    "amostras_prox_branco": None
}

print(estatisticas_raw)

# TODO: (4) Plote um histograma do mosaico utilizando pyplot




### 2 - Demosaicing e balanço de branco

Complete a função para que seja possível utilizar dois algoritmos de demosaicing (LINEAR e AHD) mantendo os demais parâmetros iguais. Depois compare o balanço de branco da câmera com o automático. Gere saída de 16 bits com `gamma=(1,1)`, sem brilho automático, para as etapas seguintes. 

Utilize a função postprocess do rawpy para construção desta etapa (referencia: https://letmaik.github.io/rawpy/api/rawpy.RawPy.html, https://letmaik.github.io/rawpy/api/rawpy.Params.html#rawpy.Params)



In [ ]:
LINEAR = rawpy.DemosaicAlgorithm.LINEAR
AHD = rawpy.DemosaicAlgorithm.AHD

# TODO: (5) Complete a função para que ela possa ser executada com demosaicing LINER e AHD e white balance = camera, daylight ou auto
# Use postprocess do raw e atribua os parametros corretos de acordo com o requisitado (https://letmaik.github.io/rawpy/api/rawpy.Params.html#rawpy.Params)
def processar_isp(caminho, algoritmo, wb_modo="camera"):
    return None


rgb16_LINEAR_CAM = processar_isp(CAMINHO_RAW, LINEAR, "camera")
rgb16_AHD_CAM = processar_isp(CAMINHO_RAW, AHD, "camera")
rgb16_AHD_WB_AUTO = processar_isp(CAMINHO_RAW, AHD, "auto")

# TODO: (6) Configure o ISP como: AHD + balanço registrado pela câmera, e atribua a imagem correspondente a rgb16_final.
rgb16_final = None
configuracao_isp = {"demosaicing": None, 
                    "balanco_branco": None}

exibir([
    ("LINEAR + WB câmera", linear_para_srgb8(rgb16_LINEAR_CAM / 65535.0)),
    ("AHD + WB câmera", linear_para_srgb8(rgb16_AHD_CAM / 65535.0)),
    ("AHD + WB automático", linear_para_srgb8(rgb16_AHD_WB_AUTO / 65535.0)),
])
print(configuracao_isp)

### 3 - Redução do ruído

Implemente uma função para melhoria da imagem removendo o ruído atraves do filtro Gaussiano e Bilateral (Dica: Use o Opencv para essas operações)

In [ ]:
rgb_linear = rgb16_final.astype(np.float32) / 65535.0

parametros_melhoria = {
    "filtro_1": "gaussiano 5x5, sigma=0.8",
    "filtro_2": "bilateral d=7, sigmaColor=0.08, sigmaSpace=7",
    "filtro_escolhido": "bilateral",
}

# TODO: (7) - Complete a funcao para que seja possivel utilizar o filtro bilateral ou gaussiano para processar a imagem (ou que nenhum processado seja aplicado)
def aplicar_melhoria(imagem, filtro="bilateral", forca_nitidez=0.6):
    x = np.clip(np.asarray(imagem, dtype=np.float32), 0.0, 1.0)

    # TODO: gaussiana, bilateral, ou nenhum
    filtrada = None
    return filtrada

filtrado_1, = aplicar_melhoria(rgb_linear, "gaussiano", 0.0)
filtrado_2, = aplicar_melhoria(rgb_linear, "bilateral", 0.0)

exibir([
    ("Sem filtro", linear_para_srgb8(rgb_linear)),
    ("Gaussiano", linear_para_srgb8(filtrado_1)),
    ("Bilateral", linear_para_srgb8(filtrado_2)),
])
print(parametros_melhoria)

### 4 - Tons e exibição

In [ ]:
def finalizar_para_jpeg(imagem_linear, expoente_tons=0.9):
    x = np.clip(np.asarray(imagem_linear, dtype=np.float32), 0.0, 1.0)
    tons = np.power(x, float(expoente_tons))
    return linear_para_srgb8(tons)

rgb_final_8bit = finalizar_para_jpeg(filtrado_2, expoente_tons=0.9)
parametros_tons = {
    "curva": "potência sobre RGB linear",
    "expoente": 0.9,
    "codificacao_saida": "sRGB por partes",
}

exibir([("Prévia rawpy", previa_rawpy),
         ("Processado", rgb_final_8bit)])
print(parametros_tons)

### 5 - Compressão e análise

In [ ]:
qualidades = [40, 60, 80, 95]
resultados_jpeg = []

# TODO: (8) Complete a função save jpeg
# Salve as imagens para JPEG na pasta de saída com nome processado_qualidade_Q.jpg
# Substitua o Q pela qualidade passada como parametro

def save_to_jpeg(qualidade):
    pass
    caminho_jpeg = None
    return caminho_jpeg

# TODO: (8.1) - Complete o código substindo os None inseridos
for q in qualidades:
    caminho_jpeg = None
    reaberta_bgr = None
    reaberta_rgb = None
    resultados_jpeg.append({
        "qualidade": q,
        "arquivo": str(caminho_jpeg),
        "bytes": caminho_jpeg.stat().st_size,
        "imagem": reaberta_rgb,
    })

for r in resultados_jpeg:
    print(f'Q={r["qualidade"]:>2} | {r["bytes"] / 1024:.1f} KiB | {r["arquivo"]}')

exibir([(f'Q={r["qualidade"]}', r["imagem"]) for r in resultados_jpeg])

In [ ]:
# (9) - Implemente o MSE
def calcular_mse(referencia, teste):
    return None

# (10) - Implemente o PSNR
def calcular_psnr(referencia, teste, valor_maximo=255.0):
    return None

tabela_jpeg = []
for r in resultados_jpeg:
    tabela_jpeg.append({
        "qualidade": r["qualidade"],
        "KiB": round(r["bytes"] / 1024, 1),
        "MSE": round(calcular_mse(rgb_final_8bit, r["imagem"]), 3),
        "PSNR_dB": round(calcular_psnr(rgb_final_8bit, r["imagem"]), 2),
    })

for linha in tabela_jpeg:
    print(linha)

# TODO: (11) - Compare a sua imagem processada com diferentes qualidades de jpeg e balaceamento de branco com o jpeg original gerado pela camera
# Exiba os valores de mse e psnr e compare utilizando com o jpeg original como referencia




### 6 - Pipeline completo

In [ ]:
def processar_fotografia_raw(caminho_raw, configuracao):
    '''Executa o pipeline e retorna RGB uint8 e dados do experimento.'''

    caminho = Path(caminho_raw)

    if not caminho.is_file():
        raise FileNotFoundError(caminho)

    # TODO: (12) preencha os parametros de acordo com a configuracao passada 
    rgb16 = processar_isp(None,None, None)

    # TODO: (13) Utilize a operacao realizada no codigo para a transformacao linear
    linear = None

     # TODO: (14) preencha os parametros de acordo com a configuracao passada 
    melhorada = aplicar_melhoria(linear, None, None)

     # TODO: (15) preencha os parametros de acordo com a configuracao passada 
    rgb8 = finalizar_para_jpeg(None, None)

    info = {"arquivo_origem": caminho.name,
            "dimensoes_saida": tuple(rgb8.shape),
            "configuracao": {k: str(v) for k, v in configuracao.items()}}
    
    return rgb8, info

configuracao_final = {
    "demosaicing": rawpy.DemosaicAlgorithm.AHD,
    "balanco_branco": "camera",
    "filtro": "bilateral",
    "forca_nitidez": 0.6,
    "expoente_tons": 0.9,
    "qualidade_jpeg": 80,
}

rgb_pipeline, info_pipeline = processar_fotografia_raw(CAMINHO_RAW, configuracao_final)
caminho_final = PASTA_SAIDA / "fotografia_final.jpg"
ok = cv2.imwrite(
    str(caminho_final), cv2.cvtColor(rgb_pipeline, cv2.COLOR_RGB2BGR),
    [cv2.IMWRITE_JPEG_QUALITY, configuracao_final["qualidade_jpeg"]],
)

print(info_pipeline)
print("Saída:", caminho_final, "| KiB:",
      round(caminho_final.stat().st_size / 1024, 1))

exibir([("Prévia rawpy", previa_rawpy), ("Imagem final", rgb_pipeline)])


### 7 - Implementado NPR

Após o processamento da imagem, podemos realizar manipulações como a aplição de um filtro de renderização

In [ ]:
def posterizar(imagem_bgr, niveis=8):
    if not 2 <= niveis <= 64:
        raise ValueError("niveis deve estar entre 2 e 64")
    largura = 256.0 / niveis
    indices = np.floor(imagem_bgr.astype(np.float32) / largura)
    return np.clip((indices + 0.5) * largura, 0, 255).astype(np.uint8)


def filtro_npr(imagem_bgr, 
               niveis=8, 
               passes_bilateral=2,
               block_size=9, 
               C=3):

    suavizada = imagem_bgr.copy()

    # TODO: (16) aplique N (passes_bilateral) passagens de filtro bilateral em `suavizada`.
    # Sugestão inicial para testar: d=9, sigmaColor=75, sigmaSpace=75.
    suavizada = None

    # TODO: (17) aplique posterizar à imagem suavizada.
    cores = None

    # TODO: (18) crie uma versão em tons de cinza a partir da imagem original.
    cinza = None

    # TODO: (19) atenue pontos isolados antes de detectar contornos (use filtro de media com kernel 5).
    cinza_sem_ruido = None

    # TODO: (20) obtenha uma máscara binária de contornos por limiarização
    # Dica: usar cv2.adaptiveThreshold, usando block_size e C
    bordas = None

    
    resultado = cv2.bitwise_and(cores, cores, mask=bordas)
    return resultado


In [ ]:

jpeg_final_bgr = cv2.imread(str(caminho_final), cv2.IMREAD_COLOR)
if jpeg_final_bgr is None:
    raise FileNotFoundError(f"Não foi possível reabrir {caminho_final}")

npr_bgr = filtro_npr(jpeg_final_bgr, niveis=8)
npr_bgr_4, _ = filtro_npr(jpeg_final_bgr, niveis=4)
origem_rgb = cv2.cvtColor(jpeg_final_bgr, cv2.COLOR_BGR2RGB)
npr_rgb = cv2.cvtColor(npr_bgr, cv2.COLOR_BGR2RGB)
npr_rgb_4 = cv2.cvtColor(npr_bgr_4, cv2.COLOR_BGR2RGB)

exibir([
    ("JPEG final", origem_rgb),
    ("NPR: 8 níveis por canal", npr_rgb),
    ("NPR: 4 níveis por canal", npr_rgb_4),
])
